# Week 5: Casablanca Mobility Data Exploration

This notebook provides an interactive environment to explore the Casablanca map-matched trip data and prototype analytics for the TaaSim platform.

In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
import json

# Load existing data
df = pd.read_csv('../data/raw/casablanca_real_roads_final.csv')
print(f"Loaded {len(df)} trips.")
df.head()

## 1. Parse Polylines and Extract Coordinates

In [ ]:
def get_start_point(polyline_str):
    try:
        coords = json.loads(polyline_str)
        return coords[0] if coords else None
    except:
        return None

df['start_point'] = df['CASA_POLYLINE'].apply(get_start_point)
df = df.dropna(subset=['start_point'])
df[['lon', 'lat']] = pd.DataFrame(df['start_point'].tolist(), index=df.index)
df.head()

## 2. Visualize Demand Heatmap

In [ ]:
casablanca_map = folium.Map(location=[33.5731, -7.5898], zoom_start=12)
heat_data = [[row['lat'], row['lon']] for index, row in df.iterrows()]
HeatMap(heat_data).add_to(casablanca_map)
casablanca_map

## 3. Hourly Trip Volume

In [ ]:
df['hour'] = pd.to_datetime(df['TIMESTAMP'], unit='s').dt.hour
df.groupby('hour').size().plot(kind='bar', title='Trips per Hour of Day')